# Topic 06 - Finite Differences: Explicit vs Implicit vs Crank-Nicolson

This note explains the difference between the three finite difference methods used in FINM 32000.

Main course sources: Lecture 3 and Lecture 4, especially L3.7-L3.17 and L4.4-L4.15.

## 1. The Big Picture

Finite difference methods are used when an option price satisfies a partial differential equation, but we cannot or do not want to solve that equation exactly.

The option price is a function of stock price and time:

$$C(S,t).$$

A finite difference method replaces the continuous surface $C(S,t)$ with values on a grid.

For example:

$$C_j^n \approx C(S_j,t_n).$$

Here:

- $j$ indexes the stock-price or log-price grid.
- $n$ indexes the time grid.
- $C_j^n$ is the numerical approximation at that grid point.

In option pricing, we usually know the payoff at expiry and work backward in time to find today's price.

## 2. The Model PDE For Comparing The Methods

To compare explicit, implicit, and Crank-Nicolson clearly, use the simple model PDE:

$$C_t+\frac{1}{2}\sigma^2 C_{xx}=0.$$

This is the stripped-down version of the Black-Scholes PDE after removing drift and discounting complications.

The full Black-Scholes PDE has more terms, but the main difference between the finite difference schemes is already visible here.

Define:

$$\alpha=\frac{1}{2}\sigma^2\frac{\Delta t}{(\Delta x)^2}.$$

This number measures how much variance the model tries to fit into one time step relative to the grid spacing.

## 3. The Shared Ingredients

All three schemes approximate the time derivative by:

$$C_t\approx \frac{C_j^{n+1}-C_j^n}{\Delta t}.$$

All three schemes approximate the second spatial derivative by a central difference:

$$C_{xx}\approx \frac{C_{j+1}-2C_j+C_{j-1}}{(\Delta x)^2}.$$

The question is: at which time layer do we evaluate this second derivative?

That is the whole difference:

- Explicit uses the known future layer $n+1$.
- Implicit uses the unknown current layer $n$.
- Crank-Nicolson uses the average of layers $n$ and $n+1$.

Because option pricing works backward, the layer $n+1$ is already known when solving for layer $n$.

## 4. Explicit Method: Use Known Future Values

The explicit scheme uses the spatial derivative at time layer $n+1$:

$$\frac{C_j^{n+1}-C_j^n}{\Delta t}+\frac{1}{2}\sigma^2\frac{C_{j+1}^{n+1}-2C_j^{n+1}+C_{j-1}^{n+1}}{(\Delta x)^2}=0.$$

Solving directly for $C_j^n$ gives:

$$C_j^n=\alpha C_{j+1}^{n+1}+(1-2\alpha)C_j^{n+1}+\alpha C_{j-1}^{n+1}.$$

Plain English:

Today's value at one grid point is computed directly from the three neighboring values in the next time column.

So explicit finite difference feels like a trinomial tree: up, middle, down.

## 5. Explicit Method: Strengths And Weaknesses

Strength:

It is easy. Once column $n+1$ is known, each value in column $n$ is computed directly.

No linear system is needed.

Weakness:

It is only conditionally stable.

For the simple model problem, the stability condition is:

$$\alpha\le \frac{1}{2}.$$

Since:

$$\alpha=\frac{1}{2}\sigma^2\frac{\Delta t}{(\Delta x)^2},$$

this means:

$$\Delta t\le \frac{(\Delta x)^2}{\sigma^2}.$$

Plain English:

If the time step is too large relative to the space step, the method tries to push too much variance through too narrow a grid. The numerical errors can blow up.

## 6. Implicit Method: Use Unknown Current Values

The implicit scheme uses the spatial derivative at the unknown time layer $n$:

$$\frac{C_j^{n+1}-C_j^n}{\Delta t}+\frac{1}{2}\sigma^2\frac{C_{j+1}^{n}-2C_j^{n}+C_{j-1}^{n}}{(\Delta x)^2}=0.$$

Rearranging gives:

$$-\alpha C_{j+1}^n+(1+2\alpha)C_j^n-\alpha C_{j-1}^n=C_j^{n+1}.$$

Now $C_j^n$ depends on other unknown values in the same column: $C_{j+1}^n$ and $C_{j-1}^n$.

So we cannot solve each row separately.

We must solve the whole time column at once as a linear system.

## 7. Implicit Method: Strengths And Weaknesses

Strength:

It is unconditionally stable for the model problem.

That means it does not require the explicit scheme's restriction:

$$\Delta t\le \frac{(\Delta x)^2}{\sigma^2}.$$

Weakness:

It requires solving a linear system at every time step.

In practice, the matrix is usually tridiagonal, so it can be solved efficiently. But it is still more work than the explicit method.

Accuracy:

The implicit method has the same basic order of accuracy as explicit:

$$O(\Delta t)+O((\Delta x)^2).$$

So implicit is not automatically more accurate than explicit. Its main advantage is stability.

## 8. Crank-Nicolson: Average Explicit And Implicit

Crank-Nicolson uses the average of the spatial derivative at the known future layer $n+1$ and the unknown current layer $n$.

The scheme is:

$$\frac{C_j^{n+1}-C_j^n}{\Delta t}+\frac{1}{2}\cdot\frac{1}{2}\sigma^2\left[\frac{C_{j+1}^{n}-2C_j^{n}+C_{j-1}^{n}}{(\Delta x)^2}+\frac{C_{j+1}^{n+1}-2C_j^{n+1}+C_{j-1}^{n+1}}{(\Delta x)^2}\right]=0.$$

The first $\frac{1}{2}$ is from the PDE coefficient.

The second $\frac{1}{2}$ is because Crank-Nicolson averages the derivative at two time layers.

Rearranged, using $\alpha=\frac{1}{2}\sigma^2\frac{\Delta t}{(\Delta x)^2}$:

$$-\frac{\alpha}{2}C_{j+1}^n+(1+\alpha)C_j^n-\frac{\alpha}{2}C_{j-1}^n$$

$$=\frac{\alpha}{2}C_{j+1}^{n+1}+(1-\alpha)C_j^{n+1}+\frac{\alpha}{2}C_{j-1}^{n+1}.$$

Like the implicit method, this requires solving a linear system.

## 9. Crank-Nicolson: Strengths And Weaknesses

Strength 1:

Crank-Nicolson is unconditionally stable for the model problem.

Strength 2:

Crank-Nicolson has better time accuracy than explicit or implicit:

$$O((\Delta t)^2)+O((\Delta x)^2).$$

Explicit and implicit are only first-order in time:

$$O(\Delta t)+O((\Delta x)^2).$$

Weakness:

It requires solving a linear system, just like implicit.

In practice, Crank-Nicolson is often preferred because it combines stability with better time accuracy.

## 10. One-Sentence Difference

Explicit uses known future values, so it is easy but can be unstable.

Implicit uses unknown current values, so it requires solving a system but is stable.

Crank-Nicolson averages known future and unknown current values, so it requires solving a system but is stable and more accurate in time.

## 11. Summary Table From The Course

| Method | Main idea | Accuracy | Unconditionally stable? | Solve linear system? |
|---|---|---:|---:|---:|
| Explicit | Use spatial derivative at known layer $n+1$ | $O(\Delta t)+O((\Delta x)^2)$ | No | No |
| Implicit | Use spatial derivative at unknown layer $n$ | $O(\Delta t)+O((\Delta x)^2)$ | Yes | Yes |
| Crank-Nicolson | Average layers $n$ and $n+1$ | $O((\Delta t)^2)+O((\Delta x)^2)$ | Yes | Yes |

This is essentially the L4.15 comparison table.

## 12. How To Remember The Names

Explicit means the new unknown is explicitly written in terms of already known numbers.

In our backward pricing direction:

$$C_j^n=\text{known formula using column }n+1.$$

Implicit means the new unknown is only implicitly defined because it appears together with other unknowns.

In our backward pricing direction:

$$\text{matrix}\times C^n=\text{known vector from }C^{n+1}.$$

Crank-Nicolson means half explicit plus half implicit.

So:

$$\text{Crank-Nicolson}=\frac{1}{2}\text{explicit} + \frac{1}{2}\text{implicit}.$$

That is not a perfect algebraic definition, but it is the right memory hook.

## 13. Why Explicit Can Blow Up

In the simple explicit formula:

$$C_j^n=\alpha C_{j+1}^{n+1}+(1-2\alpha)C_j^{n+1}+\alpha C_{j-1}^{n+1},$$

the middle weight is:

$$1-2\alpha.$$

If $\alpha>1/2$, then this weight becomes negative.

That is a warning sign. The method is no longer acting like a sensible weighted average of neighboring values.

Course intuition: it is like a trinomial tree trying to fit too much variance into too small a spatial step.

So explicit finite difference needs a small enough time step relative to the space step.

## 14. Why Implicit Is More Stable

Implicit finite difference does not try to push all information locally through only known neighboring future values.

Instead, all unknown values in the current column are solved together.

This links each node to the whole time slice through the linear system.

That global coupling damps out unstable high-frequency numerical errors.

The cost is that we must solve a matrix equation at every time step.

## 15. Why Crank-Nicolson Is More Accurate In Time

Explicit evaluates the spatial derivative entirely at one side of the time interval.

Implicit evaluates it entirely at the other side.

Crank-Nicolson averages the two sides.

Averaging the two endpoints gives a more centered approximation over the time step.

That is why the time error improves from first order:

$$O(\Delta t)$$

to second order:

$$O((\Delta t)^2).$$

## 16. Exam-Style Recognition Questions

If a method gives you a direct formula like:

$$C_j^n=q_uC_{j+1}^{n+1}+q_mC_j^{n+1}+q_dC_{j-1}^{n+1},$$

that is explicit.

If a method gives you an equation involving:

$$C_{j-1}^n,\quad C_j^n,\quad C_{j+1}^n,$$

and you must solve all $j$ values together, that is implicit.

If a method has second derivatives from both time layers $n$ and $n+1$, with a one-half average, that is Crank-Nicolson.

## 17. Connection To The Practice Midterm CN Problem

The practice midterm asks you to reconstruct a hidden grid value using Crank-Nicolson.

The key clue is that Crank-Nicolson uses the average of the gamma approximation at two time columns.

For a PDE like:

$$C_t+\frac{1}{2}S^2\sigma^2C_{SS}=0,$$

Crank-Nicolson says:

$$\frac{C(S,t+\Delta t)-C(S,t)}{\Delta t}+\frac{1}{2}S^2\sigma^2\cdot\frac{1}{2}\left[\Gamma_{t+\Delta t}^{FD}+\Gamma_t^{FD}\right]=0.$$

So if one grid value is missing, you plug it into both:

- the time derivative term,
- the current-time gamma term,

and solve one algebra equation.

That is why Crank-Nicolson hand problems often look like: use a time difference plus an average of two second differences.

## 18. Common Mistakes

Mistake 1: Thinking implicit is more accurate than explicit.

Not necessarily. In the course summary, explicit and implicit both have accuracy:

$$O(\Delta t)+O((\Delta x)^2).$$

Implicit is better for stability, not automatically for accuracy.

Mistake 2: Forgetting that Crank-Nicolson averages the spatial derivative, not the option price itself.

Mistake 3: Forgetting the linear system.

Implicit and Crank-Nicolson require solving a system at every time step.

Mistake 4: Thinking unconditional stability means large grid steps are fine.

Unconditional stability means errors do not explode. It does not mean the approximation is accurate. You still need a fine enough grid.

Mistake 5: Mixing up time direction.

In option pricing, the terminal payoff is known at expiry, and we usually solve backward to today.

## 19. Mini Exercises

### Exercise 1

A finite difference method computes each $C_j^n$ directly from $C_{j-1}^{n+1}$, $C_j^{n+1}$, and $C_{j+1}^{n+1}$. Is it explicit, implicit, or Crank-Nicolson?

### Exercise 2

A finite difference method requires solving a tridiagonal linear system and has accuracy $O(\Delta t)+O((\Delta x)^2)$. Which method is it most likely to be?

### Exercise 3

A finite difference method uses an average of the second derivative at time levels $n$ and $n+1$. Which method is it?

### Exercise 4

For the explicit method with $\sigma=0.4$ and $\Delta x=0.1$, what condition must $\Delta t$ satisfy for stability in the simple model problem?

## 20. Mini Exercise Answers

### Answer 1

Explicit. The value is computed directly from known future-column values.

### Answer 2

Implicit. Crank-Nicolson also solves a tridiagonal system, but its time accuracy is usually $O((\Delta t)^2)$.

### Answer 3

Crank-Nicolson.

### Answer 4

The explicit stability condition is:

$$\Delta t\le \frac{(\Delta x)^2}{\sigma^2}.$$

Plug in $\Delta x=0.1$ and $\sigma=0.4$:

$$\Delta t\le \frac{0.1^2}{0.4^2}=\frac{0.01}{0.16}=0.0625.$$

So the time step must be at most $0.0625$.

## 21. What To Memorize

Explicit:

$$\text{known future column} \Rightarrow \text{direct formula} \Rightarrow \text{easy but conditionally stable}.$$

Implicit:

$$\text{unknown current column} \Rightarrow \text{linear system} \Rightarrow \text{stable but more work}.$$

Crank-Nicolson:

$$\text{average current and future columns} \Rightarrow \text{linear system} \Rightarrow \text{stable and second-order in time}.$$

Course summary table:

| Method | Accuracy | Stable without grid restriction? | Linear system? |
|---|---:|---:|---:|
| Explicit | $O(\Delta t)+O((\Delta x)^2)$ | No | No |
| Implicit | $O(\Delta t)+O((\Delta x)^2)$ | Yes | Yes |
| Crank-Nicolson | $O((\Delta t)^2)+O((\Delta x)^2)$ | Yes | Yes |